# 09 — Annotation bias / composition QC before CNV

This notebook reproduces the archived **annotation-bias and composition QC**
step on the canonical 18-cluster article branch while avoiding a full in-memory
read of the 351,427 × 15,176 full-gene object.

## Historical scientific semantics preserved

The archived executed notebook:

- used `leiden_scvi_main` as the 18-cluster canonical partition;
- summarized cluster composition by dataset, condition, patient, and sample;
- calculated top-category fractions, normalized entropy, and enrichment versus
  the global composition;
- assigned the same bias reasons, review priorities, and CNV-role suggestions;
- propagated those cluster-level decisions back to every cell;
- generated UMAP and composition QC figures;
- produced a bias-ready full-gene H5AD for the CNV branch.

No expression values are used in the bias logic. Therefore this version reads
only `obs`, `var` metadata, stored matrix shapes, and `X_umap`; the large
expression/count matrices remain on disk.

The historical dataset identifiers `DS5`/`DS13` have already been normalized to
the final project nomenclature `DS3`/`DS4`. This changes labels only, not cell
membership, fractions, entropy, enrichment, or bias decisions.

**Compatibility fix:** the decision-report loop uses a local `cnv_role_value` variable so it does not overwrite the cluster-level `cnv_role` DataFrame used by the final manifest/checkpoint cell.


## Archived execution checkpoints

The executed historical notebook reported:

```text
input object                 351427 × 15176
clusters                    18

readiness                    (11, 2)
cluster summary              (18, 46)
composition long             (2090, 6)
bias flags                   (18, 24)
CNV-role recommendations     (18, 9)
cell counts by status        (11, 5)
decision report              (18, 10)
output manifest              (11, 3)
```

The cluster-level bias status and CNV-role assignments are also checked against
the archived run before the output H5AD is published.

In [ ]:
# =========================
# 0) IMPORTS / PATHS / PARAMETERS
# =========================

import os
import shutil
import warnings
from pathlib import Path

import h5py
import numpy as np
import pandas as pd

import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt

try:
    from anndata.io import read_elem, write_elem
except ImportError:
    from anndata._io.specs import read_elem, write_elem


warnings.filterwarnings("ignore")


def _find_repository_root(start=None):
    start = (
        Path.cwd()
        if start is None
        else Path(start)
    )
    start = start.expanduser().resolve()

    for candidate in (
        start,
        *start.parents,
    ):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
        ):
            return candidate

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        _find_repository_root(),
    )
).expanduser().resolve()

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
)

INTEGRATED_DIR = (
    PROCESSED_DIR
    / "integrated"
)

METADATA_DIR = (
    PROJECT_DIR
    / "metadata"
)

FIGURES_DIR = (
    PROJECT_DIR
    / "figures"
)

LOG_DIR = (
    PROJECT_DIR
    / "logs"
)

for directory in (
    INTEGRATED_DIR,
    METADATA_DIR,
    FIGURES_DIR,
    LOG_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# Canonical annotated output of notebook 08.
ANNOTATED_H5AD_PRIMARY = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_HVG8000_r0_6_marker_based_annotation.h5ad"
)

ANNOTATED_H5AD_ALIAS = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_marker_based_annotation.h5ad"
)

ANNOTATED_H5AD = (
    ANNOTATED_H5AD_PRIMARY
    if ANNOTATED_H5AD_PRIMARY.exists()
    else ANNOTATED_H5AD_ALIAS
)


ANNOTATION_TSV = (
    METADATA_DIR
    / "annotation_marker_based_HVG8000_r0_6.tsv"
)

EVIDENCE_TSV = (
    METADATA_DIR
    / "cluster_marker_annotation_evidence_HVG8000_r0_6.tsv"
)

ANNOTATION_COUNTS_TSV = (
    METADATA_DIR
    / "annotation_marker_based_HVG8000_r0_6_cell_counts.tsv"
)


# Preserve historical downstream filenames.
OUT_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status.h5ad"
)

OUT_ALIAS_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_annotation_bias_ready.h5ad"
)

OUT_BUILDING_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status.building.h5ad"
)


CLUSTER_KEY = (
    "leiden_scvi_main"
)

PRIMARY_CLUSTER_KEY = (
    "leiden_scvi_HVG8000_r0_6"
)

VERSION_TAG = (
    "HVG8000_r0_6"
)


EXPECTED_N_CELLS = 351_427
EXPECTED_N_GENES = 15_176
EXPECTED_N_CLUSTERS = 18

EXPECTED_CLUSTER_MIN = 105
EXPECTED_CLUSTER_MAX = 61_646
EXPECTED_CLUSTER_MEDIAN = 14_336.5


# Historical output shapes.
EXPECTED_READINESS_SHAPE = (
    11,
    2,
)

EXPECTED_CLUSTER_SUMMARY_SHAPE = (
    18,
    46,
)

EXPECTED_COMPOSITION_LONG_SHAPE = (
    2_090,
    6,
)

EXPECTED_BIAS_FLAGS_SHAPE = (
    18,
    24,
)

EXPECTED_CNV_ROLE_SHAPE = (
    18,
    9,
)

EXPECTED_CELL_COUNTS_BY_STATUS_SHAPE = (
    11,
    5,
)

EXPECTED_DECISION_REPORT_SHAPE = (
    18,
    10,
)

EXPECTED_MANIFEST_SHAPE = (
    11,
    3,
)


# Output TSVs.
READINESS_TSV = (
    METADATA_DIR
    / f"annotation_bias_readiness_report_{VERSION_TAG}.tsv"
)

CLUSTER_SUMMARY_TSV = (
    METADATA_DIR
    / f"annotation_bias_cluster_summary_{VERSION_TAG}.tsv"
)

COMPOSITION_LONG_TSV = (
    METADATA_DIR
    / f"annotation_bias_composition_long_{VERSION_TAG}.tsv"
)

BIAS_FLAGS_TSV = (
    METADATA_DIR
    / f"annotation_bias_flags_{VERSION_TAG}.tsv"
)

CNV_ROLE_TSV = (
    METADATA_DIR
    / f"annotation_cnv_role_recommendations_{VERSION_TAG}.tsv"
)

CELL_COUNTS_BY_STATUS_TSV = (
    METADATA_DIR
    / f"annotation_bias_cell_counts_by_status_{VERSION_TAG}.tsv"
)

DECISION_REPORT_TSV = (
    METADATA_DIR
    / f"annotation_bias_decision_report_{VERSION_TAG}.tsv"
)

OUTPUT_MANIFEST_TSV = (
    METADATA_DIR
    / f"annotation_bias_output_manifest_{VERSION_TAG}.tsv"
)


FIG_DIR = (
    FIGURES_DIR
    / "annotation_bias_HVG8000_r0_6"
)

FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

sc.settings.figdir = str(
    FIG_DIR
)


def write_tsv_replace(
    dataframe,
    path,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if path.exists():
        path.unlink()

    dataframe.to_csv(
        path,
        sep="\t",
        index=False,
    )

    print(
        f"[SAVED] {path} "
        f"shape={dataframe.shape}"
    )


def safe_str_series(
    series,
):
    return (
        series
        .astype(str)
        .replace(
            {
                "nan": "unknown",
                "None": "unknown",
                "<NA>": "unknown",
            }
        )
        .fillna(
            "unknown"
        )
    )


def find_first_existing_column(
    columns,
    candidates,
):
    columns = list(
        columns
    )

    for candidate in candidates:
        if candidate in columns:
            return candidate

    return None


def detect_column(
    columns,
    must_include=None,
    must_not_include=None,
    fallback=None,
):
    must_include = [
        value.lower()
        for value
        in (
            must_include
            or []
        )
    ]

    must_not_include = [
        value.lower()
        for value
        in (
            must_not_include
            or []
        )
    ]

    for column in columns:
        low = str(
            column
        ).lower()

        if (
            all(
                value in low
                for value
                in must_include
            )
            and not any(
                value in low
                for value
                in must_not_include
            )
        ):
            return column

    return fallback


def stored_shape(
    node,
):
    if isinstance(
        node,
        h5py.Dataset,
    ):
        return tuple(
            int(value)
            for value
            in node.shape
        )

    shape = node.attrs.get(
        "shape"
    )

    if shape is None:
        raise KeyError(
            f"{node.name} has no stored shape."
        )

    return tuple(
        int(value)
        for value
        in shape
    )


def replace_h5_key(
    parent,
    key,
    value,
):
    if key in parent:
        del parent[
            key
        ]

    write_elem(
        parent,
        key,
        value,
    )


print(
    "Project:",
    PROJECT_DIR,
)

print(
    "Input h5ad:",
    ANNOTATED_H5AD,
)

print(
    "Input exists:",
    ANNOTATED_H5AD.exists(),
)

In [ ]:
# =========================
# 1) METADATA-ONLY INPUT VALIDATION
# =========================

if not ANNOTATED_H5AD.exists():
    raise FileNotFoundError(
        f"Annotated h5ad not found: {ANNOTATED_H5AD}"
    )


with h5py.File(
    ANNOTATED_H5AD,
    "r",
) as handle:

    if "obs" not in handle:
        raise KeyError(
            "Annotated H5AD is missing /obs."
        )

    if "var" not in handle:
        raise KeyError(
            "Annotated H5AD is missing /var."
        )

    if (
        "layers" not in handle
        or "counts"
        not in handle[
            "layers"
        ]
    ):
        raise KeyError(
            "Annotated H5AD is missing layers['counts']."
        )

    obs = read_elem(
        handle[
            "obs"
        ]
    )

    var = read_elem(
        handle[
            "var"
        ]
    )

    x_shape = stored_shape(
        handle[
            "X"
        ]
    )

    counts_shape = stored_shape(
        handle[
            "layers"
        ][
            "counts"
        ]
    )

    counts_encoding = (
        handle[
            "layers"
        ][
            "counts"
        ].attrs.get(
            "encoding-type",
            "",
        )
    )

    if isinstance(
        counts_encoding,
        bytes,
    ):
        counts_encoding = (
            counts_encoding.decode(
                "utf-8"
            )
        )

    x_scvi_shape = (
        tuple(
            int(value)
            for value
            in handle[
                "obsm"
            ][
                "X_scVI"
            ].shape
        )
        if (
            "obsm" in handle
            and "X_scVI"
            in handle[
                "obsm"
            ]
        )
        else None
    )

    x_umap = (
        np.asarray(
            handle[
                "obsm"
            ][
                "X_umap"
            ],
            dtype=np.float32,
        )
        if (
            "obsm" in handle
            and "X_umap"
            in handle[
                "obsm"
            ]
        )
        else None
    )


print(
    "Stored X shape:",
    x_shape,
)

print(
    "Stored counts shape:",
    counts_shape,
)

print(
    "Counts encoding:",
    counts_encoding,
)

print(
    "obs rows:",
    len(
        obs
    ),
)

print(
    "var rows:",
    len(
        var
    ),
)

print(
    "X_scVI shape:",
    x_scvi_shape,
)

print(
    "X_umap shape:",
    None
    if x_umap is None
    else x_umap.shape,
)


if x_shape != (
    EXPECTED_N_CELLS,
    EXPECTED_N_GENES,
):
    raise AssertionError(
        "Unexpected annotated X shape: "
        f"{x_shape}"
    )

if counts_shape != x_shape:
    raise AssertionError(
        "X and layers['counts'] shapes differ."
    )

if counts_encoding != "csr_matrix":
    raise AssertionError(
        "Expected CSR counts layer."
    )

if len(
    obs
) != EXPECTED_N_CELLS:
    raise AssertionError(
        "Unexpected cell count."
    )

if len(
    var
) != EXPECTED_N_GENES:
    raise AssertionError(
        "Unexpected gene count."
    )


if CLUSTER_KEY not in obs.columns:
    if PRIMARY_CLUSTER_KEY in obs.columns:
        obs[
            CLUSTER_KEY
        ] = (
            obs[
                PRIMARY_CLUSTER_KEY
            ]
            .astype(str)
        )

        print(
            f"[ADDED] {CLUSTER_KEY} "
            f"from {PRIMARY_CLUSTER_KEY}"
        )

    else:
        raise KeyError(
            f"Neither {CLUSTER_KEY} nor "
            f"{PRIMARY_CLUSTER_KEY} found in obs."
        )


obs[
    CLUSTER_KEY
] = (
    obs[
        CLUSTER_KEY
    ]
    .astype(str)
)


cluster_counts = (
    obs[
        CLUSTER_KEY
    ]
    .value_counts()
)


if cluster_counts.shape[0] != EXPECTED_N_CLUSTERS:
    raise AssertionError(
        "Canonical input is not 18 clusters."
    )

if int(
    cluster_counts.min()
) != EXPECTED_CLUSTER_MIN:
    raise AssertionError(
        "Canonical minimum cluster size changed."
    )

if int(
    cluster_counts.max()
) != EXPECTED_CLUSTER_MAX:
    raise AssertionError(
        "Canonical maximum cluster size changed."
    )

if not np.isclose(
    float(
        cluster_counts.median()
    ),
    EXPECTED_CLUSTER_MEDIAN,
):
    raise AssertionError(
        "Canonical median cluster size changed."
    )


# Detect annotation columns already present in obs.
obs_cols = list(
    obs.columns
)

annotation_col = detect_column(
    obs_cols,
    must_include=[
        "annotation",
        "hvg8000",
    ],
    must_not_include=[
        "confidence",
        "note",
        "review",
        "bias",
        "count",
        "composition",
    ],
)

lineage_col = detect_column(
    obs_cols,
    must_include=[
        "lineage",
        "hvg8000",
    ],
)

confidence_col = detect_column(
    obs_cols,
    must_include=[
        "confidence",
        "hvg8000",
    ],
)

note_col = detect_column(
    obs_cols,
    must_include=[
        "note",
        "hvg8000",
    ],
)


if annotation_col is None:
    annotation_col = detect_column(
        obs_cols,
        must_include=[
            "annotation",
        ],
        must_not_include=[
            "confidence",
            "note",
            "review",
            "bias",
            "count",
            "composition",
        ],
    )

if lineage_col is None:
    lineage_col = detect_column(
        obs_cols,
        must_include=[
            "lineage",
        ],
    )

if confidence_col is None:
    confidence_col = detect_column(
        obs_cols,
        must_include=[
            "confidence",
        ],
    )

if note_col is None:
    note_col = detect_column(
        obs_cols,
        must_include=[
            "note",
        ],
    )


print(
    "Detected annotation_col:",
    annotation_col,
)

print(
    "Detected lineage_col:",
    lineage_col,
)

print(
    "Detected confidence_col:",
    confidence_col,
)

print(
    "Detected note_col:",
    note_col,
)


# Historical fallback: reconstruct missing annotation metadata from TSV.
if (
    annotation_col is None
    and ANNOTATION_TSV.exists()
):
    ann = pd.read_csv(
        ANNOTATION_TSV,
        sep="\t",
    )

    ann.columns = [
        str(column)
        for column
        in ann.columns
    ]

    print(
        "[READ]",
        ANNOTATION_TSV,
        ann.shape,
    )


    cluster_col_ann = find_first_existing_column(
        ann.columns,
        [
            "cluster",
            CLUSTER_KEY,
            PRIMARY_CLUSTER_KEY,
        ],
    )

    if cluster_col_ann is None:
        raise KeyError(
            "No cluster column found in annotation TSV."
        )

    ann[
        cluster_col_ann
    ] = (
        ann[
            cluster_col_ann
        ]
        .astype(str)
    )


    ann_annotation_col = detect_column(
        ann.columns,
        must_include=[
            "annotation",
        ],
        must_not_include=[
            "confidence",
            "note",
            "count",
            "composition",
        ],
    )

    ann_lineage_col = detect_column(
        ann.columns,
        must_include=[
            "lineage",
        ],
    )

    ann_confidence_col = detect_column(
        ann.columns,
        must_include=[
            "confidence",
        ],
    )

    ann_note_col = detect_column(
        ann.columns,
        must_include=[
            "note",
        ],
    )


    annotation_col = (
        f"annotation_marker_based_{VERSION_TAG}"
    )

    lineage_col = (
        f"lineage_marker_based_{VERSION_TAG}"
    )

    confidence_col = (
        f"annotation_confidence_{VERSION_TAG}"
    )

    note_col = (
        f"annotation_note_{VERSION_TAG}"
    )


    cluster_series = (
        obs[
            CLUSTER_KEY
        ]
        .astype(str)
    )


    if ann_annotation_col:
        obs[
            annotation_col
        ] = (
            cluster_series.map(
                dict(
                    zip(
                        ann[
                            cluster_col_ann
                        ],
                        ann[
                            ann_annotation_col
                        ],
                    )
                )
            )
            .fillna(
                "unknown"
            )
        )
    else:
        obs[
            annotation_col
        ] = "unknown"


    if ann_lineage_col:
        obs[
            lineage_col
        ] = (
            cluster_series.map(
                dict(
                    zip(
                        ann[
                            cluster_col_ann
                        ],
                        ann[
                            ann_lineage_col
                        ],
                    )
                )
            )
            .fillna(
                "unknown"
            )
        )
    else:
        obs[
            lineage_col
        ] = "unknown"


    if ann_confidence_col:
        obs[
            confidence_col
        ] = (
            cluster_series.map(
                dict(
                    zip(
                        ann[
                            cluster_col_ann
                        ],
                        ann[
                            ann_confidence_col
                        ],
                    )
                )
            )
            .fillna(
                "unknown"
            )
        )
    else:
        obs[
            confidence_col
        ] = "unknown"


    if ann_note_col:
        obs[
            note_col
        ] = (
            cluster_series.map(
                dict(
                    zip(
                        ann[
                            cluster_col_ann
                        ],
                        ann[
                            ann_note_col
                        ],
                    )
                )
            )
            .fillna(
                ""
            )
        )
    else:
        obs[
            note_col
        ] = ""


for column in (
    annotation_col,
    lineage_col,
    confidence_col,
    note_col,
):
    if (
        column is not None
        and column
        in obs.columns
    ):
        obs[
            column
        ] = safe_str_series(
            obs[
                column
            ]
        )


required_metadata = [
    "core_dataset",
    "dataset_id",
    "condition",
    "patient_id",
    "sample_id",
]

for column in required_metadata:
    if column not in obs.columns:
        print(
            f"[WARNING] Missing metadata column: "
            f"{column}. Creating as unknown."
        )

        obs[
            column
        ] = "unknown"

    obs[
        column
    ] = safe_str_series(
        obs[
            column
        ]
    )


if annotation_col is None:
    raise RuntimeError(
        "Could not resolve annotation column."
    )

if lineage_col is None:
    raise RuntimeError(
        "Could not resolve lineage column."
    )

if confidence_col is None:
    raise RuntimeError(
        "Could not resolve confidence column."
    )

if note_col is None:
    raise RuntimeError(
        "Could not resolve note column."
    )


expected_dataset_levels = {
    "DS1_GSE103224",
    "DS2_GSE141383",
    "DS3_GSE182109",
    "DS4_GSE278450",
}

observed_dataset_levels = set(
    obs[
        "core_dataset"
    ]
    .astype(str)
    .unique()
)

if observed_dataset_levels != expected_dataset_levels:
    raise AssertionError(
        "Input core_dataset nomenclature is not "
        "the final DS1/DS2/DS3/DS4 scheme."
    )


readiness = pd.DataFrame(
    [
        {
            "metric":
                "n_cells",
            "value":
                EXPECTED_N_CELLS,
        },
        {
            "metric":
                "n_genes",
            "value":
                EXPECTED_N_GENES,
        },
        {
            "metric":
                "cluster_key",
            "value":
                CLUSTER_KEY,
        },
        {
            "metric":
                "n_clusters",
            "value":
                obs[
                    CLUSTER_KEY
                ].nunique(),
        },
        {
            "metric":
                "annotation_col",
            "value":
                annotation_col,
        },
        {
            "metric":
                "lineage_col",
            "value":
                lineage_col,
        },
        {
            "metric":
                "confidence_col",
            "value":
                confidence_col,
        },
        {
            "metric":
                "note_col",
            "value":
                note_col,
        },
        {
            "metric":
                "counts_layer_present",
            "value":
                True,
        },
        {
            "metric":
                "X_scVI_present",
            "value":
                x_scvi_shape is not None,
        },
        {
            "metric":
                "X_umap_present",
            "value":
                x_umap is not None,
        },
    ]
)


write_tsv_replace(
    readiness,
    READINESS_TSV,
)

display(
    readiness
)


if readiness.shape != EXPECTED_READINESS_SHAPE:
    raise AssertionError(
        "Readiness table shape changed: "
        f"{readiness.shape}"
    )


print(
    "\n[CANONICAL ANNOTATED INPUT VALIDATED]"
)

In [ ]:
# =========================
# Cluster composition summaries
# =========================

def entropy_from_counts(counts):
    arr = np.asarray(counts, dtype=float)
    total = arr.sum()
    if total <= 0:
        return 0.0
    p = arr / total
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def normalized_entropy_from_counts(counts):
    arr = np.asarray(counts, dtype=float)
    k = (arr > 0).sum()
    if k <= 1:
        return 0.0
    return float(entropy_from_counts(arr) / np.log2(k))


def top_category_info(df, cluster_col, category_col):
    tab = (
        df.groupby([cluster_col, category_col], observed=True)
        .size()
        .reset_index(name="n")
    )
    total = tab.groupby(cluster_col)["n"].transform("sum")
    tab["fraction"] = tab["n"] / total
    idx = tab.groupby(cluster_col)["fraction"].idxmax()
    top = tab.loc[idx].copy()
    top = top.rename(columns={
        category_col: f"top_{category_col}",
        "n": f"top_{category_col}_n",
        "fraction": f"top_{category_col}_fraction",
    })
    return top[[cluster_col, f"top_{category_col}", f"top_{category_col}_n", f"top_{category_col}_fraction"]]


def n_unique_by_cluster(df, cluster_col, category_col):
    out = (
        df.groupby(cluster_col, observed=True)[category_col]
        .nunique()
        .reset_index(name=f"n_{category_col}")
    )
    return out


obs_df = obs[
    [CLUSTER_KEY, annotation_col, lineage_col, confidence_col, "core_dataset", "dataset_id", "condition", "patient_id", "sample_id"]
].copy()

obs_df[CLUSTER_KEY] = obs_df[CLUSTER_KEY].astype(str)

cluster_summary = (
    obs_df.groupby([CLUSTER_KEY, annotation_col, lineage_col, confidence_col], observed=True)
    .size()
    .reset_index(name="n_cells")
)

cluster_summary["fraction_total_cells"] = cluster_summary["n_cells"] / EXPECTED_N_CELLS

# Add top category and diversity info
for cat in ["core_dataset", "dataset_id", "condition", "patient_id", "sample_id"]:
    cluster_summary = cluster_summary.merge(
        top_category_info(obs_df, CLUSTER_KEY, cat),
        on=CLUSTER_KEY,
        how="left"
    )
    cluster_summary = cluster_summary.merge(
        n_unique_by_cluster(obs_df, CLUSTER_KEY, cat),
        on=CLUSTER_KEY,
        how="left"
    )

# Entropy and global enrichment
for cat in ["core_dataset", "dataset_id", "condition", "patient_id", "sample_id"]:
    global_frac = obs_df[cat].value_counts(normalize=True).to_dict()
    ent_rows = []
    for cl, sub in obs_df.groupby(CLUSTER_KEY, observed=True):
        counts = sub[cat].value_counts()
        top_cat = counts.index[0]
        top_frac = counts.iloc[0] / counts.sum()
        global_top_frac = global_frac.get(top_cat, np.nan)
        enrich = top_frac / global_top_frac if global_top_frac and global_top_frac > 0 else np.nan
        ent_rows.append({
            CLUSTER_KEY: str(cl),
            f"{cat}_entropy": entropy_from_counts(counts.values),
            f"{cat}_norm_entropy": normalized_entropy_from_counts(counts.values),
            f"top_{cat}_global_fraction": global_top_frac,
            f"top_{cat}_enrichment_vs_global": enrich,
        })
    ent_df = pd.DataFrame(ent_rows)
    cluster_summary = cluster_summary.merge(ent_df, on=CLUSTER_KEY, how="left")

cluster_summary = cluster_summary.sort_values(
    CLUSTER_KEY,
    key=lambda s: s.astype(str).map(lambda x: int(x) if str(x).isdigit() else str(x))
).reset_index(drop=True)

write_tsv_replace(cluster_summary, CLUSTER_SUMMARY_TSV)
display(cluster_summary)


if cluster_summary.shape != EXPECTED_CLUSTER_SUMMARY_SHAPE:
    raise AssertionError(
        "Cluster-summary shape changed: "
        f"{cluster_summary.shape}"
    )

In [ ]:
# =========================
# Long-format composition table
# =========================

composition_parts = []

for cat in ["core_dataset", "dataset_id", "condition", "patient_id", "sample_id"]:
    tab = (
        obs_df.groupby([CLUSTER_KEY, cat], observed=True)
        .size()
        .reset_index(name="n_cells")
    )
    tab["variable"] = cat
    tab = tab.rename(columns={cat: "category"})
    tab["cluster_total_cells"] = tab.groupby(CLUSTER_KEY)["n_cells"].transform("sum")
    tab["fraction_within_cluster"] = tab["n_cells"] / tab["cluster_total_cells"]
    composition_parts.append(tab[[CLUSTER_KEY, "variable", "category", "n_cells", "cluster_total_cells", "fraction_within_cluster"]])

composition_long = pd.concat(composition_parts, axis=0, ignore_index=True)
composition_long = composition_long.sort_values([CLUSTER_KEY, "variable", "fraction_within_cluster"], ascending=[True, True, False])

write_tsv_replace(composition_long, COMPOSITION_LONG_TSV)
display(composition_long.head(30))


if composition_long.shape != EXPECTED_COMPOSITION_LONG_SHAPE:
    raise AssertionError(
        "Composition-long shape changed: "
        f"{composition_long.shape}"
    )

In [ ]:
# =========================
# Bias flags, review priorities, and CNV-role suggestions
# =========================

def contains_any(text, terms):
    text = str(text).lower()
    return any(t.lower() in text for t in terms)


def build_bias_reason(row):
    reasons = []

    n_cells = int(row["n_cells"])
    ann = str(row[annotation_col]).lower()
    lin = str(row[lineage_col]).lower()
    conf = str(row[confidence_col]).lower()

    if n_cells < 200:
        reasons.append("very_small_cluster_lt200")
    elif n_cells < 500:
        reasons.append("small_cluster_lt500")

    if row.get("top_dataset_id_fraction", 0) >= 0.90 and row.get("top_dataset_id_enrichment_vs_global", 0) >= 1.20:
        reasons.append("dataset_dominant")
    if row.get("top_condition_fraction", 0) >= 0.85 and row.get("top_condition_enrichment_vs_global", 0) >= 1.30:
        reasons.append("condition_dominant")
    if row.get("top_patient_id_fraction", 0) >= 0.70:
        reasons.append("patient_dominant")
    if row.get("top_sample_id_fraction", 0) >= 0.70:
        reasons.append("sample_dominant")

    if contains_any(ann + " " + lin + " " + conf, ["ambiguous", "mixed", "review", "ambient", "erythroid", "hb_high", "moderate_low", "low"]):
        reasons.append("annotation_or_marker_ambiguity")

    if contains_any(ann + " " + lin, ["putative_tumor", "tumor"]):
        reasons.append("malignancy_requires_CNV_confirmation")

    if len(reasons) == 0:
        reasons.append("no_major_bias_flag")

    return ";".join(reasons)


def assign_bias_status(reason):
    if "very_small_cluster_lt200" in reason or "erythroid" in reason or "ambient" in reason:
        return "review_or_exclude"
    if "annotation_or_marker_ambiguity" in reason:
        return "review_ambiguous"
    if "patient_dominant" in reason or "sample_dominant" in reason:
        return "patient_or_sample_dominant"
    if "dataset_dominant" in reason:
        return "dataset_dominant"
    if "condition_dominant" in reason:
        return "condition_dominant"
    return "broadly_represented"


def assign_review_priority(row):
    reason = str(row["cluster_bias_reason"])
    ann = str(row[annotation_col]).lower()
    n_cells = int(row["n_cells"])

    if "review_or_exclude" in row["cluster_bias_status"]:
        return "high"
    if contains_any(reason + " " + ann, ["ambiguous", "mixed", "ambient", "erythroid", "patient_dominant", "sample_dominant"]):
        return "high"
    if n_cells < 500 or contains_any(reason, ["dataset_dominant", "condition_dominant"]):
        return "medium"
    if contains_any(reason, ["malignancy_requires_CNV_confirmation"]):
        return "medium"
    return "standard"


def assign_cnv_role(row):
    ann = str(row[annotation_col]).lower()
    lin = str(row[lineage_col]).lower()
    conf = str(row[confidence_col]).lower()
    combined = " ".join([ann, lin, conf, str(row["cluster_bias_reason"]).lower()])

    if contains_any(combined, ["erythroid", "hb_high", "ambient", "mixed"]):
        return "exclude_or_review_before_CNV"
    if contains_any(combined, ["putative_tumor", "tumor"]):
        return "CNV_query_putative_tumor_like"
    if contains_any(combined, ["cycling"]):
        return "CNV_context_dependent_cycling_review"
    if contains_any(combined, ["oligodendrocyte", "endothelial", "t_nk", "b_plasma", "mast"]):
        return "CNV_reference_core_or_extended_normal_candidate"
    if contains_any(combined, ["myeloid", "tam", "monocyte", "macrophage"]):
        return "CNV_reference_extended_myeloid_candidate"
    if contains_any(combined, ["mural", "pericyte", "fibroblast", "stromal"]):
        return "CNV_reference_extended_stromal_review"
    return "CNV_role_review_needed"


bias_df = cluster_summary.copy()
bias_df["cluster_bias_reason"] = bias_df.apply(build_bias_reason, axis=1)
bias_df["cluster_bias_status"] = bias_df["cluster_bias_reason"].map(assign_bias_status)
bias_df["cluster_review_priority"] = bias_df.apply(assign_review_priority, axis=1)
bias_df["cnv_role_suggestion"] = bias_df.apply(assign_cnv_role, axis=1)

bias_cols = [
    CLUSTER_KEY, annotation_col, lineage_col, confidence_col, "n_cells", "fraction_total_cells",
    "cluster_bias_status", "cluster_bias_reason", "cluster_review_priority", "cnv_role_suggestion",
    "top_dataset_id", "top_dataset_id_fraction", "top_dataset_id_enrichment_vs_global",
    "top_condition", "top_condition_fraction", "top_condition_enrichment_vs_global",
    "top_patient_id", "top_patient_id_fraction",
    "top_sample_id", "top_sample_id_fraction",
    "n_dataset_id", "n_condition", "n_patient_id", "n_sample_id",
]
bias_cols = [c for c in bias_cols if c in bias_df.columns]
bias_out = bias_df[bias_cols].copy()

write_tsv_replace(bias_out, BIAS_FLAGS_TSV)

cnv_role = bias_out[[CLUSTER_KEY, annotation_col, lineage_col, confidence_col, "n_cells", "cnv_role_suggestion", "cluster_review_priority", "cluster_bias_status", "cluster_bias_reason"]].copy()
write_tsv_replace(cnv_role, CNV_ROLE_TSV)

cell_counts_by_status = (
    bias_out.groupby(["cluster_bias_status", "cluster_review_priority", "cnv_role_suggestion"], observed=True)["n_cells"]
    .sum()
    .reset_index()
    .sort_values("n_cells", ascending=False)
)
cell_counts_by_status["fraction_total_cells"] = cell_counts_by_status["n_cells"] / EXPECTED_N_CELLS
write_tsv_replace(cell_counts_by_status, CELL_COUNTS_BY_STATUS_TSV)

display(bias_out)
display(cell_counts_by_status)


if bias_out.shape != EXPECTED_BIAS_FLAGS_SHAPE:
    raise AssertionError(
        "Bias-flags shape changed: "
        f"{bias_out.shape}"
    )

if cnv_role.shape != EXPECTED_CNV_ROLE_SHAPE:
    raise AssertionError(
        "CNV-role table shape changed: "
        f"{cnv_role.shape}"
    )

if (
    cell_counts_by_status.shape
    != EXPECTED_CELL_COUNTS_BY_STATUS_SHAPE
):
    raise AssertionError(
        "Cell-counts-by-status shape changed: "
        f"{cell_counts_by_status.shape}"
    )

In [ ]:
# =========================
# 5) HISTORICAL CLUSTER-LEVEL DECISION AUDIT
# =========================

historical_bias_status = {
    "0": "broadly_represented",
    "1": "broadly_represented",
    "2": "review_ambiguous",
    "3": "broadly_represented",
    "4": "broadly_represented",
    "5": "dataset_dominant",
    "6": "broadly_represented",
    "7": "broadly_represented",
    "8": "broadly_represented",
    "9": "review_ambiguous",
    "10": "patient_or_sample_dominant",
    "11": "broadly_represented",
    "12": "review_ambiguous",
    "13": "broadly_represented",
    "14": "broadly_represented",
    "15": "dataset_dominant",
    "16": "review_or_exclude",
    "17": "review_or_exclude",
}


historical_review_priority = {
    "0": "medium",
    "1": "standard",
    "2": "standard",
    "3": "standard",
    "4": "medium",
    "5": "medium",
    "6": "standard",
    "7": "medium",
    "8": "standard",
    "9": "high",
    "10": "high",
    "11": "medium",
    "12": "high",
    "13": "standard",
    "14": "standard",
    "15": "medium",
    "16": "high",
    "17": "high",
}


historical_cnv_role = {
    "0": "CNV_query_putative_tumor_like",
    "1": "CNV_reference_extended_myeloid_candidate",
    "2": "CNV_context_dependent_cycling_review",
    "3": "CNV_reference_extended_myeloid_candidate",
    "4": "CNV_query_putative_tumor_like",
    "5": "CNV_reference_core_or_extended_normal_candidate",
    "6": "CNV_reference_extended_myeloid_candidate",
    "7": "CNV_query_putative_tumor_like",
    "8": "CNV_reference_core_or_extended_normal_candidate",
    "9": "exclude_or_review_before_CNV",
    "10": "CNV_reference_extended_myeloid_candidate",
    "11": "CNV_query_putative_tumor_like",
    "12": "CNV_reference_extended_stromal_review",
    "13": "CNV_reference_extended_stromal_review",
    "14": "CNV_reference_core_or_extended_normal_candidate",
    "15": "CNV_reference_core_or_extended_normal_candidate",
    "16": "CNV_reference_core_or_extended_normal_candidate",
    "17": "exclude_or_review_before_CNV",
}


observed_bias_status = dict(
    zip(
        bias_out[
            CLUSTER_KEY
        ].astype(str),
        bias_out[
            "cluster_bias_status"
        ].astype(str),
    )
)

observed_review_priority = dict(
    zip(
        bias_out[
            CLUSTER_KEY
        ].astype(str),
        bias_out[
            "cluster_review_priority"
        ].astype(str),
    )
)

observed_cnv_role = dict(
    zip(
        bias_out[
            CLUSTER_KEY
        ].astype(str),
        bias_out[
            "cnv_role_suggestion"
        ].astype(str),
    )
)


if observed_bias_status != historical_bias_status:
    raise RuntimeError(
        "Cluster bias-status assignments differ "
        "from the archived executed notebook."
    )

if (
    observed_review_priority
    != historical_review_priority
):
    raise RuntimeError(
        "Cluster review-priority assignments differ "
        "from the archived executed notebook."
    )

if observed_cnv_role != historical_cnv_role:
    raise RuntimeError(
        "Cluster CNV-role assignments differ "
        "from the archived executed notebook."
    )


print(
    "[HISTORICAL CLUSTER-LEVEL BIAS/CNV DECISIONS REPRODUCED]"
)

In [ ]:
# =========================
# 6) PROPAGATE BIAS/CNV LABELS TO CELL METADATA
# =========================

BIAS_STATUS_COL = (
    f"cluster_bias_status_{VERSION_TAG}"
)

BIAS_REASON_COL = (
    f"cluster_bias_reason_{VERSION_TAG}"
)

REVIEW_PRIORITY_COL = (
    f"cluster_review_priority_{VERSION_TAG}"
)

CNV_ROLE_COL = (
    f"cnv_role_suggestion_{VERSION_TAG}"
)


bias_map = dict(
    zip(
        bias_out[
            CLUSTER_KEY
        ].astype(str),
        bias_out[
            "cluster_bias_status"
        ],
    )
)

reason_map = dict(
    zip(
        bias_out[
            CLUSTER_KEY
        ].astype(str),
        bias_out[
            "cluster_bias_reason"
        ],
    )
)

priority_map = dict(
    zip(
        bias_out[
            CLUSTER_KEY
        ].astype(str),
        bias_out[
            "cluster_review_priority"
        ],
    )
)

cnv_map = dict(
    zip(
        bias_out[
            CLUSTER_KEY
        ].astype(str),
        bias_out[
            "cnv_role_suggestion"
        ],
    )
)


final_obs = (
    obs.copy()
)

cluster_series = (
    final_obs[
        CLUSTER_KEY
    ]
    .astype(str)
)


final_obs[
    BIAS_STATUS_COL
] = (
    cluster_series
    .map(
        bias_map
    )
    .fillna(
        "unknown"
    )
    .astype(str)
)

final_obs[
    BIAS_REASON_COL
] = (
    cluster_series
    .map(
        reason_map
    )
    .fillna(
        "unknown"
    )
    .astype(str)
)

final_obs[
    REVIEW_PRIORITY_COL
] = (
    cluster_series
    .map(
        priority_map
    )
    .fillna(
        "unknown"
    )
    .astype(str)
)

final_obs[
    CNV_ROLE_COL
] = (
    cluster_series
    .map(
        cnv_map
    )
    .fillna(
        "unknown"
    )
    .astype(str)
)


# Historical simple aliases for downstream notebooks.
final_obs[
    "cluster_bias_status"
] = (
    final_obs[
        BIAS_STATUS_COL
    ]
    .astype(str)
)

final_obs[
    "cluster_review_priority"
] = (
    final_obs[
        REVIEW_PRIORITY_COL
    ]
    .astype(str)
)

final_obs[
    "cnv_role_suggestion"
] = (
    final_obs[
        CNV_ROLE_COL
    ]
    .astype(str)
)


print(
    final_obs[
        [
            CLUSTER_KEY,
            annotation_col,
            BIAS_STATUS_COL,
            REVIEW_PRIORITY_COL,
            CNV_ROLE_COL,
        ]
    ].head()
)

In [ ]:
# =========================
# Decision report
# =========================

decision_rows = []

for _, row in bias_out.iterrows():
    cl = str(row[CLUSTER_KEY])
    ann = str(row[annotation_col])
    lin = str(row[lineage_col])
    conf = str(row[confidence_col])
    status = str(row["cluster_bias_status"])
    priority = str(row["cluster_review_priority"])
    cnv_role_value = str(row["cnv_role_suggestion"])
    reason = str(row["cluster_bias_reason"])
    n_cells = int(row["n_cells"])

    if cnv_role_value == "CNV_query_putative_tumor_like":
        decision = "Keep as putative tumor-like query; do not call malignant until CNV confirms."
    elif cnv_role_value == "CNV_context_dependent_cycling_review":
        decision = "Keep as cycling review cluster; determine tumor/immune origin using CNV and markers."
    elif cnv_role_value.startswith("CNV_reference"):
        decision = "Candidate non-malignant reference/extended reference for CNV, subject to bias checks."
    elif cnv_role_value == "exclude_or_review_before_CNV":
        decision = "Review carefully; likely exclude from primary CNV reference/query if ambient/mixed/RBC-like."
    else:
        decision = "Review before assigning downstream CNV role."

    decision_rows.append({
        "cluster": cl,
        "n_cells": n_cells,
        "annotation": ann,
        "lineage": lin,
        "confidence": conf,
        "cluster_bias_status": status,
        "cluster_review_priority": priority,
        "cnv_role_suggestion": cnv_role_value,
        "cluster_bias_reason": reason,
        "decision": decision,
    })

decision_report = pd.DataFrame(decision_rows)
decision_report = decision_report.sort_values(
    "cluster",
    key=lambda s: s.astype(str).map(lambda x: int(x) if str(x).isdigit() else str(x))
)

write_tsv_replace(decision_report, DECISION_REPORT_TSV)
display(decision_report)


if (
    decision_report.shape
    != EXPECTED_DECISION_REPORT_SHAPE
):
    raise AssertionError(
        "Decision-report shape changed: "
        f"{decision_report.shape}"
    )

In [ ]:
# =========================
# 8) QC FIGURES — LIGHTWEIGHT UMAPs
# =========================

umap_adata = ad.AnnData(
    X=None,
    obs=final_obs.copy(),
)

if x_umap is not None:
    umap_adata.obsm[
        "X_umap"
    ] = x_umap


if "X_umap" in umap_adata.obsm:
    color_cols = [
        CLUSTER_KEY,
        annotation_col,
        lineage_col,
        confidence_col,
        BIAS_STATUS_COL,
        REVIEW_PRIORITY_COL,
        CNV_ROLE_COL,
        "core_dataset",
        "condition",
    ]

    color_cols = [
        column
        for column
        in color_cols
        if column
        in umap_adata.obs.columns
    ]


    for column in color_cols:
        try:
            legend_loc = (
                "on data"
                if column
                == CLUSTER_KEY
                else "right margin"
            )

            sc.pl.umap(
                umap_adata,
                color=column,
                show=False,
                legend_loc=legend_loc,
                frameon=False,
                size=2,
                title=column,
            )

            output = (
                FIG_DIR
                / f"umap_{column}.png"
            )

            plt.savefig(
                output,
                dpi=180,
                bbox_inches="tight",
            )

            plt.close()

            print(
                "[SAVED]",
                output,
            )

        except Exception as error:
            print(
                "[WARNING] Could not plot UMAP for",
                column,
                repr(
                    error
                ),
            )

else:
    print(
        "[WARNING] X_umap not found; "
        "skipping UMAP plots."
    )

In [ ]:
# =========================
# QC figures: composition heatmaps
# =========================

def plot_cluster_category_fraction_heatmap(obs, cluster_col, category_col, out_png, title=None):
    tmp = (
        obs.groupby([cluster_col, category_col], observed=True)
        .size()
        .reset_index(name="n")
    )
    tmp["total"] = tmp.groupby(cluster_col)["n"].transform("sum")
    tmp["fraction"] = tmp["n"] / tmp["total"]

    mat = tmp.pivot(index=cluster_col, columns=category_col, values="fraction").fillna(0.0)

    # Sort numeric cluster labels if possible
    mat = mat.loc[sorted(mat.index, key=lambda x: int(x) if str(x).isdigit() else str(x))]

    fig_w = max(8, 0.6 * mat.shape[1] + 4)
    fig_h = max(5, 0.35 * mat.shape[0] + 2)

    plt.figure(figsize=(fig_w, fig_h))
    plt.imshow(mat.values, aspect="auto", interpolation="nearest", vmin=0, vmax=1)
    plt.colorbar(label="Fraction within cluster")
    plt.xticks(range(mat.shape[1]), mat.columns, rotation=45, ha="right")
    plt.yticks(range(mat.shape[0]), mat.index)
    plt.xlabel(category_col)
    plt.ylabel(cluster_col)
    plt.title(title or f"{category_col} composition per cluster")
    plt.tight_layout()
    plt.savefig(out_png, dpi=180, bbox_inches="tight")
    plt.close()
    print("[SAVED]", out_png)


plot_cluster_category_fraction_heatmap(
    final_obs,
    CLUSTER_KEY,
    "dataset_id",
    FIG_DIR / f"heatmap_dataset_id_composition_{VERSION_TAG}.png",
    title=f"Dataset composition per cluster — {VERSION_TAG}",
)

plot_cluster_category_fraction_heatmap(
    final_obs,
    CLUSTER_KEY,
    "condition",
    FIG_DIR / f"heatmap_condition_composition_{VERSION_TAG}.png",
    title=f"Condition composition per cluster — {VERSION_TAG}",
)

plot_cluster_category_fraction_heatmap(
    final_obs,
    CLUSTER_KEY,
    BIAS_STATUS_COL,
    FIG_DIR / f"heatmap_bias_status_by_cluster_{VERSION_TAG}.png",
    title=f"Bias status per cluster — {VERSION_TAG}",
)

In [ ]:
# =========================
# 10) COPY ANNOTATED BACKBONE + PATCH BIAS/CNV METADATA
# =========================

if OUT_BUILDING_H5AD.exists():
    OUT_BUILDING_H5AD.unlink()


input_size = int(
    ANNOTATED_H5AD
    .stat()
    .st_size
)

free_space = int(
    shutil.disk_usage(
        OUT_H5AD.parent
    ).free
)


if free_space < (
    input_size
    + 5 * 1024**3
):
    raise OSError(
        "Not enough free space to publish "
        "the bias-ready H5AD safely."
    )


print(
    "[COPY] annotated canonical backbone"
)

print(
    ANNOTATED_H5AD
)

print(
    "->"
)

print(
    OUT_BUILDING_H5AD
)


shutil.copyfile(
    ANNOTATED_H5AD,
    OUT_BUILDING_H5AD,
)


print(
    "[COPY COMPLETE]"
)


bias_provenance = {
    "cluster_key":
        CLUSTER_KEY,

    "annotation_col":
        annotation_col,

    "lineage_col":
        lineage_col,

    "confidence_col":
        confidence_col,

    "bias_status_col":
        BIAS_STATUS_COL,

    "bias_reason_col":
        BIAS_REASON_COL,

    "review_priority_col":
        REVIEW_PRIORITY_COL,

    "cnv_role_col":
        CNV_ROLE_COL,

    "main_purpose":
        (
            "cluster composition, bias flags, and "
            "CNV role suggestions before CNV preparation"
        ),

    "canonical_historical_18_cluster_branch":
        True,

    "final_project_dataset_nomenclature":
        "DS1/DS2/DS3/DS4",

    "full_expression_matrix_loaded_into_ram":
        False,
}


with h5py.File(
    OUT_BUILDING_H5AD,
    "r+",
) as handle:

    replace_h5_key(
        handle,
        "obs",
        final_obs,
    )


    if "uns" not in handle:
        write_elem(
            handle,
            "uns",
            {},
        )


    # Historical notebook removed stale rank_genes_groups objects before save.
    old_rank_keys = [
        key
        for key
        in list(
            handle[
                "uns"
            ].keys()
        )
        if str(
            key
        ).startswith(
            "rank_genes_groups"
        )
    ]

    for key in old_rank_keys:
        del handle[
            "uns"
        ][
            key
        ]

        print(
            "[REMOVED old uns key]",
            key,
        )


    replace_h5_key(
        handle[
            "uns"
        ],
        f"annotation_bias_qc_{VERSION_TAG}",
        bias_provenance,
    )


    handle.flush()


print(
    "[BIAS/CNV PATCH COMPLETE]"
)

In [ ]:
# =========================
# 11) VALIDATE + ATOMICALLY PUBLISH
# =========================

check = sc.read_h5ad(
    OUT_BUILDING_H5AD,
    backed="r",
)

try:
    if check.shape != (
        EXPECTED_N_CELLS,
        EXPECTED_N_GENES,
    ):
        raise AssertionError(
            "Bias-ready H5AD shape changed."
        )

    if "counts" not in check.layers:
        raise AssertionError(
            "Bias-ready H5AD lost layers['counts']."
        )

    for column in (
        BIAS_STATUS_COL,
        BIAS_REASON_COL,
        REVIEW_PRIORITY_COL,
        CNV_ROLE_COL,
        "cluster_bias_status",
        "cluster_review_priority",
        "cnv_role_suggestion",
    ):
        if column not in check.obs.columns:
            raise AssertionError(
                f"Bias-ready H5AD is missing {column!r}."
            )


    final_cluster_counts = (
        check.obs[
            CLUSTER_KEY
        ]
        .astype(str)
        .value_counts()
    )


    if final_cluster_counts.shape[0] != EXPECTED_N_CLUSTERS:
        raise AssertionError(
            "Bias-ready H5AD lost the canonical 18 clusters."
        )


    observed_bias_status = dict(
        zip(
            bias_out[
                CLUSTER_KEY
            ].astype(str),
            bias_out[
                "cluster_bias_status"
            ].astype(str),
        )
    )

    if observed_bias_status != historical_bias_status:
        raise AssertionError(
            "Bias-ready cluster-status audit failed."
        )


    observed_datasets = set(
        check.obs[
            "core_dataset"
        ]
        .astype(str)
        .unique()
    )


    if observed_datasets != {
        "DS1_GSE103224",
        "DS2_GSE141383",
        "DS3_GSE182109",
        "DS4_GSE278450",
    }:
        raise AssertionError(
            "Final dataset nomenclature changed."
        )


    print(
        "[VALIDATED BUILD]"
    )

    print(
        "shape:",
        check.shape,
    )

    print(
        "counts layer:",
        "counts"
        in check.layers,
    )

    print(
        "canonical clusters:",
        final_cluster_counts.shape[0],
    )

    print(
        "bias statuses:",
        sorted(
            check.obs[
                BIAS_STATUS_COL
            ]
            .astype(str)
            .unique()
            .tolist()
        ),
    )

finally:
    check.file.close()


# Remove an obsolete alias first so it cannot keep an old inode alive.
if OUT_ALIAS_H5AD.exists():
    OUT_ALIAS_H5AD.unlink()


if OUT_H5AD.exists():
    OUT_H5AD.unlink()


os.replace(
    OUT_BUILDING_H5AD,
    OUT_H5AD,
)


print(
    "[PUBLISHED]"
)

print(
    OUT_H5AD
)


alias_created = False

try:
    os.link(
        OUT_H5AD,
        OUT_ALIAS_H5AD,
    )

    alias_created = True

    print(
        "[HARD-LINK ALIAS CREATED]"
    )

    print(
        OUT_ALIAS_H5AD
    )

except OSError as error:
    print(
        "[INFO] Hard-link alias was not created:"
    )

    print(
        repr(
            error
        )
    )

    print(
        "Canonical bias-ready H5AD remains:"
    )

    print(
        OUT_H5AD
    )

In [ ]:
# =========================
# 12) OUTPUT MANIFEST / FINAL CHECKPOINTS
# =========================

manifest_items = [
    (
        "input_h5ad",
        ANNOTATED_H5AD,
    ),
    (
        "output_h5ad",
        OUT_H5AD,
    ),
    (
        "output_alias_h5ad",
        OUT_ALIAS_H5AD,
    ),
    (
        "readiness_report",
        READINESS_TSV,
    ),
    (
        "cluster_summary",
        CLUSTER_SUMMARY_TSV,
    ),
    (
        "composition_long",
        COMPOSITION_LONG_TSV,
    ),
    (
        "bias_flags",
        BIAS_FLAGS_TSV,
    ),
    (
        "cnv_role_recommendations",
        CNV_ROLE_TSV,
    ),
    (
        "cell_counts_by_status",
        CELL_COUNTS_BY_STATUS_TSV,
    ),
    (
        "decision_report",
        DECISION_REPORT_TSV,
    ),
    (
        "figures_dir",
        FIG_DIR,
    ),
]


manifest = pd.DataFrame(
    [
        {
            "item":
                name,

            "path":
                str(
                    path
                ),

            "exists":
                Path(
                    path
                ).exists(),
        }
        for name, path
        in manifest_items
    ]
)


write_tsv_replace(
    manifest,
    OUTPUT_MANIFEST_TSV,
)

display(
    manifest
)


if manifest.shape != EXPECTED_MANIFEST_SHAPE:
    raise AssertionError(
        "Output-manifest shape changed: "
        f"{manifest.shape}"
    )


print(
    "\nDONE — canonical historical annotation-bias QC "
    "reproduced with memory-safe execution."
)

print(
    "Readiness:",
    readiness.shape,
)

print(
    "Cluster summary:",
    cluster_summary.shape,
)

print(
    "Composition long:",
    composition_long.shape,
)

print(
    "Bias flags:",
    bias_out.shape,
)

print(
    "CNV roles:",
    cnv_role.shape,
)

print(
    "Cell counts by status:",
    cell_counts_by_status.shape,
)

print(
    "Decision report:",
    decision_report.shape,
)

print(
    "Bias-ready H5AD:",
    OUT_H5AD,
)

print(
    "Alias created:",
    alias_created,
)

## Expected successful completion

```text
Readiness:              (11, 2)
Cluster summary:        (18, 46)
Composition long:       (2090, 6)
Bias flags:             (18, 24)
CNV roles:              (18, 9)
Cell counts by status:  (11, 5)
Decision report:        (18, 10)

canonical clusters:     18
```

Historical cluster-level bias/CNV decisions must reproduce exactly. The
DS5/DS13 → DS3/DS4 nomenclature update affects category labels only.

Primary downstream H5AD:

```text
data/processed/integrated/
GBM_core_4datasets_fullgene_marker_based_annotation_bias_status.h5ad
```

Compatibility downstream path:

```text
GBM_core_4datasets_fullgene_annotation_bias_ready.h5ad
```

The compatibility path is created as a filesystem hard link when supported, so
it does not consume another full multi-gigabyte copy.

No full expression matrix is materialized in RAM.